# Simple: Email Field Analysis with PAMOLA.CORE

**Goal**: Learn email field profiling basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze email fields for validation and patterns using execute()
- Compare before/after results
- Understand domain distributions and privacy risks

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/analyzers/
        └── 01_email_analyzer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.email import EmailOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with records including email addresses suitable for profiling

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with various email patterns
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'email': [
            'alice.smith@gmail.com', 'bob.jones@yahoo.com', 'charlie.brown@outlook.com',
            'diana.prince@company.com', 'eve.adams@gmail.com', 'frank.miller@yahoo.com',
            'grace.lee@hotmail.com', 'henry.wilson@company.com', 'iris.chen@gmail.com',
            'jack.taylor@outlook.com', 'karen.white@company.com', 'leo.martin@gmail.com',
            'mia.davis@yahoo.com', 'nathan.garcia@hotmail.com', 'olivia.martinez@gmail.com',
            'paul.rodriguez@company.com', 'quinn.hernandez@outlook.com', 'rachel.lopez@gmail.com',
            'sam.gonzalez@yahoo.com', 'tina.wilson@company.com'
        ],
        'name': [
            'Alice Smith', 'Bob Jones', 'Charlie Brown', 'Diana Prince', 'Eve Adams',
            'Frank Miller', 'Grace Lee', 'Henry Wilson', 'Iris Chen', 'Jack Taylor',
            'Karen White', 'Leo Martin', 'Mia Davis', 'Nathan Garcia', 'Olivia Martinez',
            'Paul Rodriguez', 'Quinn Hernandez', 'Rachel Lopez', 'Sam Gonzalez', 'Tina Wilson'
        ],
        'department': [
            'Sales', 'Marketing', 'IT', 'HR', 'Sales',
            'Marketing', 'IT', 'HR', 'Sales', 'Marketing',
            'IT', 'HR', 'Sales', 'Marketing', 'IT',
            'HR', 'Sales', 'Marketing', 'IT', 'HR'
        ]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure field name** for processing

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "email"  # Customize this!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "email"  # Field to analyze - CUSTOMIZE THIS!
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to process: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Sample values: {list(df[field_name].unique()[:5])}")
print(f"  Data type: {df[field_name].dtype}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="email_profiling",
    description=f"Profiling of '{field_name}' email field",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create EmailOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `field_name=field_name`: Field to analyze
- `top_n=20`: Number of top domains to include in results
- `min_frequency=1`: Minimum frequency for domain dictionary inclusion
- `analyze_privacy_risk=True`: Assess privacy risks from email patterns
- `profile_type='email'`: Profile type for organizing artifacts
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = EmailOperation(
    # Core parameters
    field_name=field_name,
    
    # Analysis parameters
    top_n=20,                        # Top N domains to include
    min_frequency=1,                 # Min count for domain dictionary
    analyze_privacy_risk=True,       # Assess privacy risks
    profile_type='email',            # Profile type for organization
    
    # Processing settings
    chunk_size=1000,
    use_dask=False,
    use_vectorization=False,
    parallel_processes=1,
    npartitions=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,     # Create visualization artifacts
    save_output=True,                # Save results to files
    force_recalculation=False        # Use cache when available
)

print("✓ Operation configured")
print(f"  Field:              {operation.field_name}")
print(f"  Top N domains:      {operation.top_n}")
print(f"  Min frequency:      {operation.min_frequency}")
print(f"  Privacy analysis:   {operation.analyze_privacy_risk}")
print(f"  Visualizations:     {operation.generate_visualization}")
print(f"  Save output:        {operation.save_output}")
print(f"  Force recalc:       {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing email field analysis...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the analysis output from task_dir
- Review metrics and statistics
- Examine domain distributions and patterns

**Generated artifacts:**
- Statistics JSON in output/
- Domain dictionary (CSV and JSON) in dictionaries/ and output/
- Privacy risk assessment JSON in output/
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load statistics file
output_files = list(task_dir.glob('output/*_stats_*.json'))
if output_files:
    stats_file = output_files[0]
    
    with open(stats_file, 'r') as f:
        stats = json.load(f)
    
    print("\n" + "=" * 80)
    print("📊 EMAIL FIELD ANALYSIS RESULTS")
    print("=" * 80)
    
    # Basic statistics
    print("\n📈 Data Quality Metrics:")
    print(f"  Total records:      {stats.get('total_rows', 0):,}")
    print(f"  Null count:         {stats.get('null_count', 0):,}")
    print(f"  Null percentage:    {stats.get('null_percentage', 0):.2f}%")
    print(f"  Valid emails:       {stats.get('valid_count', 0):,}")
    print(f"  Valid percentage:   {stats.get('valid_percentage', 0):.2f}%")
    print(f"  Invalid emails:     {stats.get('invalid_count', 0):,}")
    print(f"  Invalid percentage: {stats.get('invalid_percentage', 0):.2f}%")
    
    # Domain statistics
    print("\n📧 Domain Statistics:")
    print(f"  Unique domains:     {stats.get('unique_domains', 0)}")
    
    # Top domains
    if 'top_domains' in stats and stats['top_domains']:
        print("\n📊 Top Email Domains:")
        for domain, count in list(stats['top_domains'].items())[:10]:
            bar = '█' * min(50, count * 2)
            pct = (count / stats.get('valid_count', 1)) * 100
            print(f"  {domain:30s} {bar:20s} {count:2d} ({pct:5.1f}%)")
    
    # Personal patterns
    if 'personal_patterns' in stats:
        patterns = stats['personal_patterns']
        print("\n👤 Email Pattern Analysis:")
        print(f"  Valid emails analyzed: {patterns.get('total_valid_emails', 0)}")
        
        if 'pattern_percentages' in patterns:
            print("\n  Pattern Detection:")
            for pattern_name, percentage in list(patterns['pattern_percentages'].items())[:5]:
                if percentage > 0:
                    print(f"    {pattern_name.replace('_', ' ').title():30s}: {percentage:.1f}%")
    
    # Display result metrics
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    if result.metrics:
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Email field '{field_name}' successfully analyzed!")
else:
    print("⚠️  No statistics file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Statistics and privacy assessment JSON
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
├── dictionaries/     # Domain dictionary CSV
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'dictionaries', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance and effectiveness metrics
2. **Statistics Data**: Email field analysis with validation and patterns
3. **Domain Dictionary**: Frequency distribution of email domains
4. **Privacy Risk Assessment**: Privacy risk analysis based on email uniqueness
5. **Visualizations**: Charts showing domain distributions

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display key metrics
            print("\n📊 KEY METRICS:")
            metric_keys = ['total_records', 'null_count', 'null_percentage', 
                          'valid_count', 'valid_percentage', 'unique_domains']
            
            for key in metric_keys:
                if key in metrics:
                    value = metrics[key]
                    if isinstance(value, float):
                        print(f"   {key:25s}: {value:.2f}")
                    else:
                        print(f"   {key:25s}: {value}")
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. STATISTICS DATA (NEWEST)
print("\n\n2️⃣ STATISTICS DATA:")
print("=" * 80)

output_dir = task_dir / "output"

if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    stats_files = sorted(
        output_dir.glob("*_stats_*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )

    if not stats_files:
        print("⚠️  No statistics files found")
    else:
        latest_stats_file = stats_files[0]
        mtime = datetime.fromtimestamp(latest_stats_file.stat().st_mtime)

        print(f"📄 File     : {latest_stats_file.name}")
        print(f"🕒 Modified : {mtime:%Y-%m-%d %H:%M:%S}")
        print(f"📦 Size     : {latest_stats_file.stat().st_size / 1024:.1f} KB\n")

        try:
            with open(latest_stats_file, "r", encoding="utf-8") as f:
                stats = json.load(f)

            total = stats.get("total_rows", 0)
            valid = stats.get("valid_count", 0)
            invalid = stats.get("invalid_count", 0)
            nulls = stats.get("null_count", 0)

            # BASIC VALIDATION SUMMARY
            print("📧 EMAIL VALIDATION SUMMARY")
            print("-" * 80)
            print(f"Total rows     : {total:,}")
            print(f"Valid emails   : {valid:,} ({stats.get('valid_percentage', 0):.1f}%)")
            print(f"Invalid emails : {invalid:,} ({stats.get('invalid_percentage', 0):.1f}%)")
            print(f"Null values    : {nulls:,} ({stats.get('null_percentage', 0):.1f}%)")

            if valid == 0:
                print("\n⚠️  No valid emails detected – skipping further analysis")
                raise SystemExit

            # DOMAIN DISTRIBUTION (RE-IDENTIFICATION SIGNAL)
            if "top_domains" in stats:
                print("\n🌐 DOMAIN DISTRIBUTION (TOP 10)")
                print("-" * 80)

                for domain, count in list(stats["top_domains"].items())[:10]:
                    pct = (count / valid) * 100
                    bar = "█" * min(30, int(pct))
                    print(f"{domain:25s}: {bar:<30} {pct:5.1f}% ({count})")

                unique_domains = stats.get("unique_domains", 0)
                print(f"\nUnique domains: {unique_domains}")

                if unique_domains <= 10:
                    print("⚠️  Low domain diversity – higher linkage risk")

            # PERSONAL IDENTIFIABLE PATTERNS
            patterns = stats.get("personal_patterns", {})
            if patterns:
                print("\n👤 PERSONAL EMAIL PATTERN ANALYSIS")
                print("-" * 80)

                pattern_pct = patterns.get("pattern_percentages", {})
                high_risk_patterns = 0

                for pattern, pct in sorted(
                    pattern_pct.items(),
                    key=lambda x: x[1],
                    reverse=True,
                ):
                    label = pattern.replace("_", " ").title()
                    print(f"{label:35s}: {pct:5.1f}%")

                    if pct > 20:
                        high_risk_patterns += 1

                if high_risk_patterns > 0:
                    print(
                        "\n🚨 High prevalence of name-based email patterns detected\n"
                        "   → Emails are likely personally identifiable\n"
                        "   → Strong recommendation: MASK or HASH this column"
                    )

            # FINAL PRIVACY ASSESSMENT
            print("\n🛡️ PRIVACY ASSESSMENT")
            print("-" * 80)

            pii_risk = (
                valid > 0
                and stats.get("unique_domains", 0) <= 10
                and any(pct > 20 for pct in patterns.get("pattern_percentages", {}).values())
            )

            if pii_risk:
                print("❌ HIGH RISK – Direct Identifier (Email)")
                print("   Recommended actions:")
                print("   • Mask local-part (e.g. j***@domain.com)")
                print("   • Or apply irreversible hash")
                print("   • Exclude from analytics outputs")
            else:
                print("⚠️  Medium risk – review before exposure")

        except Exception as e:
            print(f"❌ Error reading statistics: {e}")

# 3. DOMAIN DICTIONARY (NEWEST)
print("\n\n3️⃣ DOMAIN DICTIONARY:")
print("-" * 80)

dict_dir = task_dir / "dictionaries"

if not dict_dir.exists():
    print(f"⚠️  Dictionaries directory not found: {dict_dir}")
else:
    dict_files = sorted(
        dict_dir.glob("*domains_dictionary*.csv"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )

    if not dict_files:
        print("⚠️  No domain dictionary files found")
    else:
        latest_dict_file = dict_files[0]
        mtime = datetime.fromtimestamp(latest_dict_file.stat().st_mtime)

        print(f"📄 File     : {latest_dict_file.name}")
        print(f"🕒 Modified : {mtime:%Y-%m-%d %H:%M:%S}\n")

        try:
            dict_df = pd.read_csv(latest_dict_file)

            # BASIC STATS
            total_domains = len(dict_df)
            total_pct = dict_df["percentage"].sum()

            print("📊 DOMAIN DICTIONARY SUMMARY")
            print("-" * 80)
            print(f"Total unique domains : {total_domains}")
            print(f"Coverage (%)         : {total_pct:.2f}%\n")

            # TOP DOMAINS
            print("🌐 TOP DOMAINS (BY SHARE)")
            print("-" * 80)

            for _, row in dict_df.head(15).iterrows():
                bar = "█" * min(40, int(row["percentage"]))
                print(
                    f"{row['domain']:20s}: "
                    f"{bar:<40} {row['percentage']:5.2f}% ({int(row['count'])})"
                )

            # CONCENTRATION ANALYSIS
            top_3_pct = dict_df.head(3)["percentage"].sum()
            top_5_pct = dict_df.head(5)["percentage"].sum()

            print("\n📈 DOMAIN CONCENTRATION")
            print("-" * 80)
            print(f"Top 3 domains share : {top_3_pct:.2f}%")
            print(f"Top 5 domains share : {top_5_pct:.2f}%")

            if top_5_pct > 70:
                print("🚨 HIGH CONCENTRATION – strong linkage risk")
            elif top_5_pct > 50:
                print("⚠️  Medium concentration – review recommended")
            else:
                print("✅ Healthy distribution")

            # FREE EMAIL PROVIDER CHECK
            free_domains = {
                "gmail.com",
                "yahoo.com",
                "outlook.com",
                "hotmail.com",
                "icloud.com",
                "proton.me",
                "mail.com",
            }

            dict_df["is_free_provider"] = dict_df["domain"].isin(free_domains)
            free_pct = dict_df.loc[
                dict_df["is_free_provider"], "percentage"
            ].sum()

            print("\n📨 FREE EMAIL PROVIDERS")
            print("-" * 80)
            print(f"Free provider share : {free_pct:.2f}%")

            if free_pct > 80:
                print("🚨 Mostly personal email addresses detected")
            elif free_pct > 60:
                print("⚠️  Majority personal email usage")
            else:
                print("✅ Significant corporate email presence")

            # PRIVACY INTERPRETATION
            print("\n🛡️ PRIVACY INTERPRETATION")
            print("-" * 80)

            if free_pct > 60 and top_5_pct > 50:
                print("❌ HIGH PRIVACY RISK")
                print("   Evidence:")
                print("   • Email domains dominated by free providers")
                print("   • High concentration in few domains")
                print("   Impact:")
                print("   • Strong re-identification potential")
                print("   Recommendation:")
                print("   • Mask or hash email column")
                print("   • Avoid exposing domain-level analytics publicly")
            else:
                print("⚠️  Moderate risk – controlled exposure only")

        except Exception as e:
            print(f"❌ Error reading domain dictionary: {e}")

# 4. PRIVACY RISK ASSESSMENT (NEWEST)
print("\n\n4️⃣ PRIVACY RISK ASSESSMENT:")
print("-" * 80)

if not output_dir.exists():
    print("⚠️  Output directory not found")
else:
    privacy_files = sorted(
        output_dir.glob("*privacy_risk*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )

    if not privacy_files:
        print("ℹ️  No privacy risk assessment found (may be disabled)")
    else:
        latest_privacy_file = privacy_files[0]
        mtime = datetime.fromtimestamp(latest_privacy_file.stat().st_mtime)

        print(f"📄 File     : {latest_privacy_file.name}")
        print(f"🕒 Modified : {mtime:%Y-%m-%d %H:%M:%S}\n")

        try:
            with open(latest_privacy_file, "r", encoding="utf-8") as f:
                privacy = json.load(f)

            # BASIC METRICS
            total = privacy.get("total_valid_emails", 0)
            unique = privacy.get("unique_emails", 0)
            uniqueness_ratio = privacy.get("uniqueness_ratio", 0.0)
            singleton_count = privacy.get("singleton_count", 0)
            singleton_pct = privacy.get("singleton_percentage", 0.0)
            most_freq = privacy.get("most_frequent_count", 0)

            print("🔒 BASIC PRIVACY METRICS")
            print("-" * 80)
            print(f"Field name            : {privacy.get('field_name', 'N/A')}")
            print(f"Total valid values    : {total:,}")
            print(f"Unique values         : {unique:,}")
            print(f"Uniqueness ratio      : {uniqueness_ratio:.2%}")
            print(f"Singleton count       : {singleton_count:,}")
            print(f"Singleton percentage : {singleton_pct:.2f}%")
            print(f"Most frequent count   : {most_freq:,}")

            # RISK RE-EVALUATION (DO NOT TRUST RAW risk_level)
            print("\n⚠️  RISK RE-EVALUATION")
            print("-" * 80)

            recalculated_risk = "Low"
            risk_reasons = []

            # Email is direct identifier
            risk_reasons.append("Email is a DIRECT IDENTIFIER")

            if singleton_pct >= 20:
                recalculated_risk = "High"
                risk_reasons.append("High singleton percentage (≥20%)")
            elif singleton_pct >= 10:
                recalculated_risk = "High"
                risk_reasons.append("Moderate-high singleton percentage (≥10%)")

            if uniqueness_ratio >= 0.3:
                recalculated_risk = "High"
                risk_reasons.append("High uniqueness ratio (≥30%)")

            print(f"Declared risk level   : {privacy.get('risk_level', 'N/A')}")
            print(f"Recalculated risk     : 🚨 {recalculated_risk}")

            print("\nReasons:")
            for r in risk_reasons:
                print(f"  • {r}")

            # EXAMPLES (STRICTLY LIMITED)
            examples = privacy.get("most_frequent_examples", {})

            if examples:
                print("\n📌 Most frequent values (LIMITED DISPLAY)")
                print("-" * 80)
                print("⚠️  Displaying frequency only – values SHOULD be masked")

                for i, (val, cnt) in enumerate(examples.items()):
                    if i >= 5:
                        break
                    masked = val.split("@")[0][:2] + "***@" + val.split("@")[1]
                    print(f"  {masked:35s}: {cnt}")

            # FINAL PRIVACY VERDICT
            print("\n🛡️ FINAL PRIVACY VERDICT")
            print("-" * 80)

            if recalculated_risk == "High":
                print("❌ HIGH PRIVACY RISK")
                print("Impact:")
                print("  • High re-identification probability")
                print("  • Direct exposure of personal identifiers")
                print("Recommendations:")
                print("  • Hash or tokenize email column")
                print("  • Disable raw value exposure")
                print("  • Exclude from k-anonymity quasi sets")
                print("  • Restrict access to authorized roles only")
            else:
                print("⚠️  MODERATE RISK")
                print("Recommendations:")
                print("  • Mask before sharing")
                print("  • Limit aggregation granularity")

        except Exception as e:
            print(f"❌ Error reading privacy assessment: {e}")

# 5. VISUALIZATIONS (NEWEST BATCH)
print("\n\n5️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                viz_name = viz_file.stem.replace('_', ' ').replace(field_name, '').strip().title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure email profiling with full parameters  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Email field analysis validates format and extracts domains
- Domain distributions reveal email provider patterns
- Personal pattern detection identifies name-based email structures
- Privacy risk assessment evaluates uniqueness and singleton rates
- High uniqueness ratios indicate potential re-identification risks
- Domain dictionaries useful for data standardization and cleaning

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_email_analyzer_advanced.ipynb`** includes:
- Parallel processing for large datasets
- Vectorized operations for performance
- Custom domain filtering and grouping
- Advanced privacy risk metrics
- Email normalization strategies

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)